# Lesson 5.6 — Developer Productivity and Operational Enablement

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Configure Claude tools and Claude Code for a team
- Design AI-assisted developer workflows with guardrails
- Support debugging and operational issue resolution with Claude

**You will produce:** A team enablement guide and an AI-assisted debugging workflow.

## Concepts

Domain 7 (7%) is about enabling *teams* to build and operate with Claude efficiently. **Claude Code** is Anthropic's agentic coding tool that works in the terminal/IDE; configuring it for a team means shared conventions, project context files, permitted tools/commands, and guardrails (what it may touch, where secrets live, review requirements).

**AI-assisted developer workflows** embed Claude in the SDLC: scaffolding, code review, test generation, documentation, and incident triage — always with human review on merges and a clear boundary around production actions. **Operational support**: Claude can accelerate debugging by summarizing logs, proposing hypotheses (feeding the Lesson 4.4 decision tree), and drafting runbook steps. The architect's job is to set this up so it raises velocity *without* eroding the governance from the rest of this phase. Verify Claude Code setup details in the official docs.

> **Reference:** [Anthropic — Claude Code](https://docs.anthropic.com/en/docs/claude-code/overview)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Draft a team enablement configuration

We capture the team-level settings and guardrails as structured config: shared context, permitted actions, and review rules. This is the policy an org standardizes on.

In [ ]:
team_config = {
  "shared_context": ["project README","coding standards","architecture_doc.md"],
  "permitted_tools": ["read/edit repo files","run tests","lint"],
  "restricted": ["no prod deploys","no secret access","no DB writes without review"],
  "review_rules": ["human approves all merges","CI + eval gate must pass"],
  "secrets": "via env/secret manager, never in prompts or repo",
}
with open("team_enablement.json","w",encoding="utf-8") as fh:
    json.dump(team_config, fh, indent=2)
for k,v in team_config.items(): print(f"{k}: {v}")
# See https://docs.anthropic.com for Claude Code configuration specifics.

## Code-Along: Step 2 — Build an AI-assisted debugging workflow

We wire Claude into operational triage: summarize a failing log, propose a hypothesis, and hand off to the Phase 4 diagnosis tree. Mock-safe so it runs without a key.

In [ ]:
incident_log = ("ERROR retrieval returned 0 chunks for 40% of requests after 02:00 reindex; "
                "model/latency unchanged; users report confident wrong answers")
summary = ask_claude(f"Summarize this incident and propose the single most likely cause:\n{incident_log}",
                     model=MODELS['haiku'], max_tokens=120)
print("Claude triage:", summary)

# Hand off to the deterministic diagnosis tree from Lesson 4.4
signals = {"confident_but_wrong": True, "doc_or_index_changed": True, "retrieval_miss_up": True}
def quick_diagnose(s):
    if s.get('doc_or_index_changed') or s.get('retrieval_miss_up'):
        return "retrieval/indexing — inspect reindex + embeddings (Sample 3)"
    return "gather more signals"
print("Decision-tree cause:", quick_diagnose(signals))

## Exercise

Design a **safe code-review workflow** with Claude Code: list the steps (context load, diff review, test generation, human approval) and the guardrails at each step (no prod actions, secrets handling, required human sign-off). Then write a short policy sentence on where the human boundary is non-negotiable and why.

In [ ]:
# TODO: workflow = [ {step, guardrail}, ... ]
# TODO: print the workflow and the non-negotiable human boundary policy

## Key Takeaways

- Configure Claude Code for teams with shared context, permitted tools, and review rules.
- Embed AI in the SDLC but keep human approval on merges and production actions.
- Claude accelerates incident triage; hand off to deterministic diagnosis for decisions.
- Enablement must raise velocity without eroding governance — secrets never in prompts.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.